In [ ]:
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar
import cv2
import numpy as np

camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True
car = NvidiaRacecar()

In [ ]:
def find_lane_center(image):
    # Crop ROI — bottom portion of frame
    roi = image[150:224, :]
    
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)
    
    # Split ROI into left and right halves
    h, w = binary.shape
    mid = w // 2
    left_half = binary[:, :mid]
    right_half = binary[:, mid:]
    
    # Find centroid of white pixels in each half
    M_left = cv2.moments(left_half)
    M_right = cv2.moments(right_half)
    
    left_cx = int(M_left["m10"] / M_left["m00"]) if M_left["m00"] > 0 else None
    right_cx = int(M_right["m10"] / M_right["m00"]) + mid if M_right["m00"] > 0 else None
    
    # Compute lane center
    if left_cx is not None and right_cx is not None:
        return (left_cx + right_cx) // 2
    elif left_cx is not None:
        # Only left line visible — assume lane center is to its right
        return left_cx + 60  # estimate based on typical lane width in pixels
    elif right_cx is not None:
        return right_cx - 60
    else:
        return None

In [ ]:
def compute_error(cx, image_width=224, offset=0):
    return cx - (image_width / 2) - offset

In [ ]:
Kp = 0.02  # increased from 0.005
Kd = 0.002

prev_error = 0

def steering_from_error(error):
    global prev_error
    derivative = error - prev_error
    prev_error = error
    steer = Kp * error + Kd * derivative
    return max(-1.0, min(1.0, steer))

In [ ]:
import time

prev_error = 0  # reset before every run

car.throttle = -0.2

try:
    while True:
        image = camera.value
        cx = find_lane_center(image)
        if cx is not None:
            error = compute_error(cx, offset= -20)
            car.steering = -steering_from_error(error)
        else:
            car.steering = 0.0
        time.sleep(0.03)
except KeyboardInterrupt:
    car.throttle = 0.0
    car.steering = 0.0

In [ ]:
car.throttle = 0.0 # stop

In [ ]:
image = camera.value
roi = image[150:224, :]
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)

# Display the binary image
import ipywidgets
from jetcam.utils import bgr8_to_jpeg
binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
widget = ipywidgets.Image(format='jpeg')
widget.value = bgr8_to_jpeg(binary_bgr)
display(widget)

# And print the centroid
M = cv2.moments(binary)
if M["m00"] != 0:
    cx = int(M["m10"] / M["m00"])
    print(f"Centroid x: {cx} (frame center is 112)")

In [ ]:
image = camera.value
roi = image[150:224, :]
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, binary = cv2.threshold(blur, 180, 255, cv2.THRESH_BINARY)

h, w = binary.shape
mid = w // 2
left_half = binary[:, :mid]
right_half = binary[:, mid:]

M_left = cv2.moments(left_half)
M_right = cv2.moments(right_half)
left_cx = int(M_left["m10"] / M_left["m00"]) if M_left["m00"] > 0 else None
right_cx = int(M_right["m10"] / M_right["m00"]) + mid if M_right["m00"] > 0 else None

print(f"Left line cx: {left_cx}")
print(f"Right line cx: {right_cx}")
print(f"Lane center: {(left_cx + right_cx)//2 if left_cx and right_cx else 'N/A'}")
print(f"Frame center: 112")

In [ ]:
# Simulate "lane is to the right of camera" (centroid > 112)
test_error = compute_error(150)  # positive error
print(f"Error: {test_error}")
print(f"Steering output: {-steering_from_error(test_error)}")
car.steering = -steering_from_error(test_error)
# Watch the wheels — which way do they turn?

In [ ]:
car.steering = 0.0

In [ ]:
car.steering_offset = 1   # try this first
car.steering = 0.0
car.throttle = -0.2
# Let it roll forward for 2 seconds
import time
time.sleep(5)
car.throttle = 0.0